### Category Types
| Type               | Example           | Encoding      |
| ------------------ | ----------------- | ------------- |
| Nominal (no order) | City, Color       | One-Hot       |
| Ordinal (ordered)  | Low, Medium, High | Ordinal       |
| High Cardinality   | Zipcode, User ID  | Target / Hash |
| Binary             | Yes / No          | 0 / 1         |

In [1]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Mumbai", "Delhi", "Bangalore", "Mumbai", "Delhi"],
    "education": ["High School", "Bachelor", "Master", "Bachelor", "High School"],
    "purchased": [0, 1, 1, 0, 1]
})
df

,city,education,purchased
0,Mumbai,High School,0
1,Delhi,Bachelor,1
2,Bangalore,Master,1
3,Mumbai,Bachelor,0
4,Delhi,High School,1


### Label Encoding

In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["city_encoded"] = le.fit_transform(df["city"])
df

,city,education,purchased,city_encoded
0,Mumbai,High School,0,2
1,Delhi,Bachelor,1,1
2,Bangalore,Master,1,0
3,Mumbai,Bachelor,0,2
4,Delhi,High School,1,1


**This is dangerous:**   
Model thinks:    
```
Delhi < Mumbai < Bangalore
```

**Rule:**    
Label Encoding is safe **only** for:
- Tree-based models
- Ordinal variables (with correct order)

### One-Hot Encoding
#### Pandas One-Hot

In [3]:
pd.get_dummies(df, columns=["city"])

,education,purchased,city_encoded,city_Bangalore,city_Delhi,city_Mumbai
0,High School,0,2,False,False,True
1,Bachelor,1,1,False,True,False
2,Master,1,0,True,False,False
3,Bachelor,0,2,False,False,True
4,High School,1,1,False,True,False


- No false ordering
- Works with linear models
- High dimensionality risk

#### sklearn One-Hot

In [11]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop="first")
encoded = ohe.fit_transform(df[["city"]])
encoded

array([[0., 1.],
       [1., 0.],
       [0., 0.],
       [0., 1.],
       [1., 0.]])

Drop One Dummy
- Avoids multicollinearity in linear models
- Not needed for trees

The dropped category becomes the **reference category.**

### Ordinal Encoding

In [14]:
from sklearn.preprocessing import OrdinalEncoder

order = [["High School", "Bachelor", "Master"]]

oe = OrdinalEncoder(categories=order)
df["education_encoded"] = oe.fit_transform(df[["education"]])
df["education_encoded"]

0    0.0
1    1.0
2    2.0
3    1.0
4    0.0
Name: education_encoded, dtype: float64

- Preserves meaning
- Required for ordered categories

### Binary Encoding

In [ ]:
df["is_delhi"] = (df["city"] == "Delhi").astype(int)

- Useful for strong binary signals
- Feature engineering goldmine

### High Cardinality Categories
Example: `zipcode`, `user_id`
- One-hot explodes dimensions
- Label encoding lies

**Temporary Safe Options**
- Frequency encoding
- Target encoding
- Hashing trick

In [16]:
df["city_freq"] = df["city"].map(df["city"].value_counts())

### Encoding vs Algorithm
| Algorithm         | Safe Encoding     |
| ----------------- | ----------------- |
| Linear / Logistic | One-Hot           |
| KNN / SVM         | One-Hot + Scaling |
| Decision Trees    | Label / Ordinal   |
| Random Forest     | Label / Ordinal   |
| XGBoost           | Label / Target    |